# Bank's Request Practice

### You work in a bank's IT department. The reporting department needs all transaction dates in a standard YYYY-MM-DD format to generate monthly reports.

### Objective: Convert all dates currently in various formats (2026/08/01, 01-08-2026, etc.) into a uniform format and handle invalid dates.

# Execute Spark Local Session

In [50]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

# Import all libraries you need

In [51]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import *
from pyspark.sql.dataframe import DataFrame
from pyspark.sql import functions as F
from datetime import datetime

# Create Spark Session Function

In [52]:
def create_spark_session(app_name: str = "BankRequest") -> SparkSession:
    """
    Create a SparkSession with the specified application name.

    Args:
        app_name (str): The name of the Spark application. Default is "BankRequest".

    Returns:
        SparkSession: A SparkSession object.
    """
    spark = (
        SparkSession.builder
        .appName(app_name)
        .config("spark.sql.shuffle.partitions", "2")
        .config("spark.default.parallelism", "4")
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )
    return spark

# Create Dataframe from Bank CSV File

In [53]:
def create_dataframe(spark: SparkSession, file_path: str) -> DataFrame:
    """
    Create a DataFrame with data from CSV dataset_bank file

    Args:
        spark (SparkSession): A SparkSession object.

    Returns:
        DataFrame: A DataFrame containing the data from the CSV dataset_bank file and schema structured below.
    """
    schema = StructType([
        StructField("transaction_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("transaction_date", StringType(), True),
        StructField("transaction_type", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("account_number", StringType(), True),
        StructField("branch_name", StringType(), True),
        StructField("employee_id", StringType(), True),
        StructField("customer_name", StringType(), True),
        StructField("email", StringType(), True),
        StructField("phone", IntegerType(), True),
        StructField("account_status", StringType(), True),
        StructField("risk_level", StringType(), True),
        StructField("product_type", StringType(), True),
        StructField("channel", StringType(), True),
        StructField("country", StringType(), True),
        StructField("state", StringType(), True),
        StructField("balance", DoubleType(), True),
        StructField("credit_score", IntegerType(), True),
        StructField("last_login", IntegerType(), True)
    ])

    """
    Create a DataFrame with data from CSV dataset_bank file and structured schema defined above.
    """
    df = spark.read.csv(
        file_path, 
        header=True, 
        schema = schema, 
        sep=","
        ) 
    
    return df

# Select only data needed

In [121]:
def select_date_and_transform_columns(df: DataFrame) -> DataFrame:
    """
    Selects the transaction_id and transaction_date columns from the given DataFrame.

    Args:
        df (DataFrame): A DataFrame containing transaction data.
    Returns:
        DataFrame: A DataFrame with only the date columns.
    """
    df_selected = df.select("transaction_id", "transaction_date")
    
    # 1. Normalize replacing '/' to '-'
    raw_date = regexp_replace(col("transaction_date"), "/", "-")
    
    # 2. Trying parsing data with no lit all date formats and coalesce to get the first non-null value
    parsed_date = coalesce(
        try_to_date(raw_date, "yyyy-MM-dd"),
        try_to_date(raw_date, "dd-MM-yyyy"),
        try_to_date(raw_date, "MM-dd-yyyy"),
        try_to_date(raw_date, "yyyy-M-d")
    )
    
    # 3. Default date with no NULL's
    default_date = try_to_date(lit("1900-01-01"), "yyyy-MM-dd")
    final_date = coalesce(parsed_date, default_date)
    
    # 4. Transforming to final column with format yyyy-MM-dd
    df_selected = df_selected.withColumn(
        "transaction_date_cleaned",
        date_format(final_date, "yyyy-MM-dd")
    )
    
    return df_selected

# Main Orchestate Function

In [122]:
def main():
    """
    Main function to create an Orchestrate all ETL process and detect errors through the same process.
    """
    spark = create_spark_session()
    try:
        print(" -------- Original DataFrame --------")
        df = create_dataframe(spark, "C:/temp/dataset_bank.csv")
        df.show(5, truncate=False)
        
        print(" -------- Date Columns DataFrame --------")
        df_date_columns = select_date_and_transform_columns(df)
        df_date_columns.show(100, truncate=False)
        
    except Exception as Error:
        print(f"Error occurred: {Error}")
        spark.stop()

if __name__ == "__main__":
    main()

 -------- Original DataFrame --------
+--------------+-----------+----------------+----------------+--------+--------------+-----------+-----------+-------------------+---------------------+-----+--------------+----------+------------+----------+-------------+-----+--------+------------+----------+
|transaction_id|customer_id|transaction_date|transaction_type|amount  |account_number|branch_name|employee_id|customer_name      |email                |phone|account_status|risk_level|product_type|channel   |country      |state|balance |credit_score|last_login|
+--------------+-----------+----------------+----------------+--------+--------------+-----------+-----------+-------------------+---------------------+-----+--------------+----------+------------+----------+-------------+-----+--------+------------+----------+
|TXN00000368   |CUST00328  |NULL            |Transfer        |NULL    |ACC0000000104 |SAN DIEGO  |NULL       |JENNIFER GARCIA    |NULL                 |NULL |Suspended     |low